In [2]:
import numpy as np
import pandas as pd
import os

In [ ]:
def generate_dataset(v_raw_path, w_raw_path, ds_name, output_dir='./dataset/PeMSD7/'):

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    dist_matrix = pd.read_csv(w_raw_path, header=None).values

    sigma = 10 
    epsilon = 0.5
    
    # exp(-d^2 / sigma^2)
    w_weighted = np.exp(-(dist_matrix**2) / (sigma**2))
    w_weighted[w_weighted < epsilon] = 0

    np.fill_diagonal(w_weighted, 0)

    pd.DataFrame(w_weighted).to_csv(os.path.join(output_dir, f'W_{ds_name}_P.csv'), header=None, index=None)
    print(f"Saved: {output_dir}W_{ds_name}_P.csv")

    v_data = pd.read_csv(v_raw_path, header=None)

    v_data.to_csv(os.path.join(output_dir, f'V_{ds_name}.csv'), header=None, index=None)
    print(f"Saved: {output_dir}V_{ds_name}.csv")

if __name__ == '__main__':
    dct_dt = {
        "228": ['./dataset/PeMSD7/PeMSD7_V_228.csv', './dataset/PeMSD7/PeMSD7_W_228.csv'],
        "1026": ['./dataset/PeMSD7/PeMSD7_V_1026.csv', './dataset/PeMSD7/PeMSD7_W_1026.csv']
    }

    for key, paths in dct_dt.items():
        print(f"Processing {key}")
        generate_dataset(*paths, key)

## Test the shape of data for splitting

In [5]:
def check_dataset_lengths():
    dct_dt = {
        "AirQuality": ['../dataset/AirQuality/air_quality_adj.npy', '../dataset/AirQuality/air_quality_data.npy'],
        "Shanghai": ['../dataset/ShanghaiRailway/W_43_P.csv', '../dataset/ShanghaiRailway/V_43.csv'],
        "PeMSD7_228": ['../dataset/PeMSD7/W_228_P.csv', '../dataset/PeMSD7/V_228.csv'],
        "PeMSD7_1026": ['../dataset/PeMSD7/W_1026_P.csv', '../dataset/PeMSD7/V_1026.csv']
    }

    steps_per_day_map = {
        "AirQuality": 24,       
        "Shanghai": 1440,       
        "PeMSD7_228": 288,      
        "PeMSD7_1026": 288      
    }

    print("="*50)
    print("Dataset Length and Days Check")
    print("="*50)

    for ds_name, paths in dct_dt.items():
        v_path = paths[1] 
        
        if not os.path.exists(v_path):
            print(f"\n[{ds_name}] File not found: {v_path}")
            continue
            
        try:
            if v_path.endswith('.csv'):
                data = pd.read_csv(v_path, header=None).values
            else:
                data = np.load(v_path)
                
            total_steps = data.shape[0]
            nodes = data.shape[1] if len(data.shape) > 1 else "Unknown"
    
            steps_per_day = steps_per_day_map.get(ds_name, 288)
            total_days = total_steps / steps_per_day
            
            print(f"\n[{ds_name}]")
            print(f"  - Read path: {v_path}")
            print(f"  - Data shape: {data.shape} (Time steps: {total_steps}, Nodes: {nodes})")
            print(f"  - Default frequency: {steps_per_day} steps per day")
            print(f"  - Converted days: Total approx {total_days:.2f} days")
            
            if total_days < 44:
                print("  Warning: Data length is less than 44 days! Can only use ratio split (e.g., 70/10/20).")
            else:
                print("  Status: Data length is sufficient, can perfectly execute 34/5/5 days split.")
                
        except Exception as e:
            print(f"\n[{ds_name}] Read failed: {e}")

if __name__ == "__main__":
    check_dataset_lengths()

Dataset Length and Days Check

[AirQuality]
  - Read path: ../dataset/AirQuality/air_quality_data.npy
  - Data shape: (35064, 12, 1) (Time steps: 35064, Nodes: 12)
  - Default frequency: 24 steps per day
  - Converted days: Total approx 1461.00 days
  Status: Data length is sufficient, can perfectly execute 34/5/5 days split.

[Shanghai]
  - Read path: ../dataset/ShanghaiRailway/V_43.csv
  - Data shape: (15999, 43) (Time steps: 15999, Nodes: 43)
  - Default frequency: 1440 steps per day
  - Converted days: Total approx 11.11 days

[PeMSD7_228]
  - Read path: ../dataset/PeMSD7/V_228.csv
  - Data shape: (12672, 228) (Time steps: 12672, Nodes: 228)
  - Default frequency: 288 steps per day
  - Converted days: Total approx 44.00 days
  Status: Data length is sufficient, can perfectly execute 34/5/5 days split.

[PeMSD7_1026]
  - Read path: ../dataset/PeMSD7/V_1026.csv
  - Data shape: (12672, 1026) (Time steps: 12672, Nodes: 1026)
  - Default frequency: 288 steps per day
  - Converted days: 